<a href="https://colab.research.google.com/github/amit-sw/colab_notebooks/blob/main/hvac_langchain_deep_agent_colab_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HVAC RAG + Deep Agent Demo (Colab)

This notebook demonstrates three answer modes over an existing OpenAI-hosted HVAC document vector store:

- **Baseline LLM**: no retrieval
- **RAG**: retrieval-grounded answer using retrieved HVAC documentation
- **Deep Agent + RAG**: a LangChain Deep Agent that can infer context from the user query, optionally use climate context when a location is mentioned, and then retrieve supporting HVAC documentation

## Setup

1. Run the install cell.
2. Set `OPENAI_API_KEY` and `OPENAI_VECTOR_STORE_ID` in Colab Secrets or environment variables.
3. Run the config cell.
4. Run the app cell and enter a free-form query.

Expected secret names:

- `OPENAI_API_KEY`
- `OPENAI_VECTOR_STORE_ID`

The notebook is presenter-oriented: it shows the query, retrieved evidence, any climate profile used by the Deep Agent, a concise agent step trace, and the final answer.

In [ ]:
# Colab install cell
%pip -q install -U openai langchain langchain-openai deepagents ipywidgets pandas

In [ ]:
import json
import os

try:
    from google.colab import userdata
except ImportError:
    userdata = None


def get_secret(name: str, default: str = "") -> str:
    if os.getenv(name):
        return os.getenv(name)
    if userdata is not None:
        try:
            value = userdata.get(name)
            if value:
                return value
        except Exception:
            pass
    return default


OPENAI_API_KEY = get_secret("OPENAI_API_KEY")
OPENAI_VECTOR_STORE_ID = get_secret("OPENAI_VECTOR_STORE_ID", "vs_your_vector_store_id")
CHAT_MODEL = os.getenv("CHAT_MODEL", "gpt-4.1-mini")
AGENT_MODEL = os.getenv("AGENT_MODEL", "openai:gpt-5.4")
DEFAULT_TOP_K = int(os.getenv("TOP_K", "4"))

if not OPENAI_API_KEY:
    raise ValueError("Missing OPENAI_API_KEY. Set it in Colab Secrets or environment variables.")

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

print(json.dumps({
    "CHAT_MODEL": CHAT_MODEL,
    "AGENT_MODEL": AGENT_MODEL,
    "OPENAI_VECTOR_STORE_ID": OPENAI_VECTOR_STORE_ID,
    "DEFAULT_TOP_K": DEFAULT_TOP_K,
}, indent=2))

if OPENAI_VECTOR_STORE_ID == "vs_your_vector_store_id":
    print("Update OPENAI_VECTOR_STORE_ID before running retrieval-backed modes.")

In [ ]:
# Core demo app cell
import json
import re
import textwrap
from datetime import UTC, datetime
from urllib.parse import urlencode
from urllib.request import Request, urlopen

import ipywidgets as widgets
import pandas as pd
from IPython.display import Markdown, clear_output, display
from deepagents import create_deep_agent
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from openai import OpenAI

openai_client = OpenAI(api_key=OPENAI_API_KEY)
baseline_llm = ChatOpenAI(model=CHAT_MODEL, temperature=0)
rag_llm = ChatOpenAI(model=CHAT_MODEL, temperature=0)

TRACE_LOG = []
LAST_CLIMATE_PROFILE = None


def add_trace(message: str) -> None:
    timestamp = datetime.now().strftime("%H:%M:%S")
    TRACE_LOG.append(f"{timestamp} | {message}")


def message_to_text(message) -> str:
    content = getattr(message, "content", message)
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for block in content:
            if isinstance(block, str):
                parts.append(block)
            elif isinstance(block, dict):
                if block.get("type") == "text":
                    parts.append(block.get("text", ""))
                elif "text" in block:
                    parts.append(str(block["text"]))
        return "\n".join(part for part in parts if part).strip()
    return str(content)


def to_plain_dict(obj):
    if isinstance(obj, dict):
        return obj
    if hasattr(obj, "model_dump"):
        return obj.model_dump()
    if hasattr(obj, "__dict__"):
        return obj.__dict__
    return {}


def http_get_json(base_url: str, params: dict, headers: dict | None = None):
    url = f"{base_url}?{urlencode(params)}"
    request = Request(url, headers=headers or {})
    with urlopen(request) as response:
        return json.loads(response.read().decode("utf-8"))


def c_to_f(value):
    if value is None or pd.isna(value):
        return None
    return round((float(value) * 9 / 5) + 32, 1)


def mm_to_inches(value):
    if value is None or pd.isna(value):
        return None
    return round(float(value) / 25.4, 1)


def compact_list(items):
    return [item for item in items if item]


US_STATE_NAMES = {
    "alabama", "alaska", "arizona", "arkansas", "california", "colorado", "connecticut", "delaware",
    "florida", "georgia", "hawaii", "idaho", "illinois", "indiana", "iowa", "kansas", "kentucky",
    "louisiana", "maine", "maryland", "massachusetts", "michigan", "minnesota", "mississippi", "missouri",
    "montana", "nebraska", "nevada", "new hampshire", "new jersey", "new mexico", "new york",
    "north carolina", "north dakota", "ohio", "oklahoma", "oregon", "pennsylvania", "rhode island",
    "south carolina", "south dakota", "tennessee", "texas", "utah", "vermont", "virginia", "washington",
    "west virginia", "wisconsin", "wyoming", "district of columbia"
}


def candidate_location_queries(location: str):
    cleaned = " ".join(location.replace(",", " ").split())
    variants = []

    def add_variant(value):
        value = " ".join(value.split())
        if value and value not in variants:
            variants.append(value)

    add_variant(location.strip())
    add_variant(cleaned)

    tokens = cleaned.split()
    if len(tokens) >= 2:
        for tail_size in (3, 2, 1):
            if len(tokens) <= tail_size:
                continue
            state_part = " ".join(tokens[-tail_size:]).lower()
            city_part = " ".join(tokens[:-tail_size])
            if state_part in US_STATE_NAMES:
                add_variant(f"{city_part}, {state_part.title()}")
                add_variant(f"{city_part}, {state_part.title()}, USA")
                break
        add_variant(f"{tokens[0]}, {' '.join(tokens[1:])}")
        add_variant(f"{tokens[0]}, {' '.join(tokens[1:])}, USA")

    return variants


def geocode_location(location: str):
    attempted = []
    for candidate in candidate_location_queries(location):
        attempted.append(candidate)
        geo = http_get_json(
            "https://geocoding-api.open-meteo.com/v1/search",
            {"name": candidate, "count": 3, "language": "en", "format": "json"},
        )
        matches = geo.get("results") or []
        if matches:
            match = matches[0]
            match["_geocoder"] = "open_meteo"
            return match, attempted

    for candidate in candidate_location_queries(location):
        nominatim_attempt = f"nominatim:{candidate}"
        attempted.append(nominatim_attempt)
        geo = http_get_json(
            "https://nominatim.openstreetmap.org/search",
            {"q": candidate, "format": "jsonv2", "limit": 1},
            headers={"User-Agent": "HVACAdvisorDemo/1.0 (Colab notebook demo)"},
        )
        if geo:
            match = geo[0]
            return {
                "name": match.get("name") or candidate.split(",")[0].strip(),
                "admin1": match.get("display_name", ""),
                "country": "",
                "latitude": float(match["lat"]),
                "longitude": float(match["lon"]),
                "_geocoder": "nominatim",
            }, attempted
    return None, attempted


def normalize_search_results(search_response):
    payload = to_plain_dict(search_response)
    raw_items = payload.get("data") or getattr(search_response, "data", []) or []
    normalized = []
    for item in raw_items:
        row = to_plain_dict(item)
        content_blocks = row.get("content") or []
        text_parts = []
        for block in content_blocks:
            block_dict = to_plain_dict(block)
            if block_dict.get("type") == "text" and block_dict.get("text"):
                text_parts.append(block_dict["text"])
        snippet = "\n\n".join(text_parts).strip()
        normalized.append({
            "file_id": row.get("file_id", ""),
            "filename": row.get("filename") or row.get("file_name") or "unknown",
            "score": round(float(row.get("score", 0.0)), 4),
            "attributes": row.get("attributes") or {},
            "snippet": snippet,
        })
    return normalized


def search_hvac_docs(query: str, top_k: int = DEFAULT_TOP_K):
    add_trace(f"Retrieval invoked for query: {query}")
    if OPENAI_VECTOR_STORE_ID == "vs_your_vector_store_id":
        warning = "OPENAI_VECTOR_STORE_ID is still a placeholder, so retrieval-backed modes cannot run yet."
        add_trace(warning)
        return {"query": query, "results": [], "warning": warning, "evidence_text": ""}

    kwargs = {
        "vector_store_id": OPENAI_VECTOR_STORE_ID,
        "query": query,
        "max_num_results": int(top_k),
        "rewrite_query": True,
    }

    try:
        response = openai_client.vector_stores.search(**kwargs)
        results = normalize_search_results(response)
        add_trace(f"Documents found: {len(results)} chunk(s) from vector store {OPENAI_VECTOR_STORE_ID}")
    except Exception as exc:
        warning = f"Retrieval failed: {exc}"
        add_trace(warning)
        return {"query": query, "results": [], "warning": warning, "evidence_text": ""}

    evidence_lines = []
    for idx, item in enumerate(results, start=1):
        snippet = item["snippet"] or "[No text snippet returned]"
        evidence_lines.append(f"[{idx}] {item['filename']} | score={item['score']}\n{snippet}")

    warning = ""
    if not results:
        warning = "No matching chunks were retrieved. The answer should explicitly say evidence was insufficient."
        add_trace(warning)

    return {
        "query": query,
        "results": results,
        "warning": warning,
        "evidence_text": "\n\n".join(evidence_lines),
    }


def build_climate_notes(annual_typical_high_f, annual_typical_low_f, hottest_month_avg_high_f, coldest_month_avg_low_f, annual_precip_in):
    notes = []
    if hottest_month_avg_high_f is not None and hottest_month_avg_high_f >= 90:
        notes.append("Hot summers suggest strong peak cooling demand and careful condenser performance review.")
    elif hottest_month_avg_high_f is not None and hottest_month_avg_high_f >= 80:
        notes.append("Warm summers make cooling efficiency and controls strategy relevant.")

    if coldest_month_avg_low_f is not None and coldest_month_avg_low_f <= 32:
        notes.append("Freezing winter conditions suggest freeze protection and cold-weather startup planning.")
    elif coldest_month_avg_low_f is not None and coldest_month_avg_low_f <= 45:
        notes.append("Cool winter conditions may matter for heating capacity and shoulder-season control logic.")

    if annual_precip_in is not None and annual_precip_in >= 50:
        notes.append("Higher annual precipitation can correlate with moisture exposure and condensate management needs.")
    elif annual_precip_in is not None and annual_precip_in <= 15:
        notes.append("A drier climate may reduce moisture-related issues but can increase dust loading and filter attention.")

    if annual_typical_high_f is not None and annual_typical_low_f is not None:
        notes.append(f"Typical daily temperatures center around {annual_typical_high_f} F highs and {annual_typical_low_f} F lows across the year.")

    return notes


def build_maintenance_notes(hottest_month_avg_high_f, coldest_month_avg_low_f, annual_precip_in):
    notes = []
    if hottest_month_avg_high_f is not None and hottest_month_avg_high_f >= 90:
        notes.append("Prioritize condenser coil cleaning, refrigerant charge verification, and peak-season diagnostics before summer.")
    if coldest_month_avg_low_f is not None and coldest_month_avg_low_f <= 32:
        notes.append("Include freeze-stat checks, drain-pan protection, and low-ambient control verification before winter.")
    if annual_precip_in is not None and annual_precip_in >= 50:
        notes.append("Increase attention on condensate drains, corrosion-prone components, and moisture-related inspections.")
    if annual_precip_in is not None and annual_precip_in <= 15:
        notes.append("Check filters and outdoor-air sections more often for dust accumulation in dry conditions.")
    if not notes:
        notes.append("Use normal manufacturer-prescribed maintenance intervals; climate data does not indicate a strong adjustment.")
    return notes


def get_climate_profile(location: str) -> dict:
    """Resolve a location with Open-Meteo and return a compact climate profile plus maintenance implications."""
    global LAST_CLIMATE_PROFILE
    location = location.strip()
    add_trace(f"Climate tool invoked for location: {location}")
    profile = {
        "location_input": location,
        "resolved_location": "",
        "lat": None,
        "lon": None,
        "annual_typical_high_f": None,
        "annual_typical_low_f": None,
        "hottest_month_avg_high_f": None,
        "coldest_month_avg_low_f": None,
        "annual_precip_in": None,
        "climate_notes": [],
        "maintenance_notes": [],
        "warning": "",
    }

    if not location:
        profile["warning"] = "No location text was supplied to the climate tool."
        LAST_CLIMATE_PROFILE = profile
        add_trace(profile["warning"])
        return profile

    try:
        match, attempted = geocode_location(location)
        if not match:
            profile["warning"] = f"Open-Meteo could not resolve the location '{location}'. Tried: {', '.join(attempted)}"
            LAST_CLIMATE_PROFILE = profile
            add_trace(profile["warning"])
            return profile

        add_trace(f"Geocoder retries: {', '.join(attempted)}")
        add_trace(f"Geocoder provider: {match.get('_geocoder', 'unknown')}")
        profile["resolved_location"] = ", ".join(compact_list([match.get("name"), match.get("admin1"), match.get("country")]))
        profile["lat"] = round(float(match["latitude"]), 4)
        profile["lon"] = round(float(match["longitude"]), 4)
        add_trace(f"Location resolved: {profile['resolved_location']}")

        current_year = datetime.now(UTC).year
        end_year = current_year - 1
        start_year = end_year - 9
        archive = http_get_json(
            "https://archive-api.open-meteo.com/v1/archive",
            {
                "latitude": profile["lat"],
                "longitude": profile["lon"],
                "start_date": f"{start_year}-01-01",
                "end_date": f"{end_year}-12-31",
                "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum",
                "timezone": "auto",
            },
        )

        daily = archive.get("daily") or {}
        if not daily.get("time"):
            profile["warning"] = f"Open-Meteo returned no climate history for {profile['resolved_location']}."
            LAST_CLIMATE_PROFILE = profile
            add_trace(profile["warning"])
            return profile

        climate_df = pd.DataFrame({
            "date": pd.to_datetime(daily["time"]),
            "tmax_c": daily["temperature_2m_max"],
            "tmin_c": daily["temperature_2m_min"],
            "precip_mm": daily.get("precipitation_sum", [None] * len(daily["time"])),
        })
        climate_df["month"] = climate_df["date"].dt.month

        monthly = climate_df.groupby("month", as_index=False).agg(
            avg_high_c=("tmax_c", "mean"),
            avg_low_c=("tmin_c", "mean"),
        )

        profile["annual_typical_high_f"] = c_to_f(climate_df["tmax_c"].mean())
        profile["annual_typical_low_f"] = c_to_f(climate_df["tmin_c"].mean())
        profile["hottest_month_avg_high_f"] = c_to_f(monthly["avg_high_c"].max())
        profile["coldest_month_avg_low_f"] = c_to_f(monthly["avg_low_c"].min())
        annual_precip_mm = climate_df.groupby(climate_df["date"].dt.year)["precip_mm"].sum().mean()
        profile["annual_precip_in"] = mm_to_inches(annual_precip_mm)
        profile["climate_notes"] = build_climate_notes(
            profile["annual_typical_high_f"],
            profile["annual_typical_low_f"],
            profile["hottest_month_avg_high_f"],
            profile["coldest_month_avg_low_f"],
            profile["annual_precip_in"],
        )
        profile["maintenance_notes"] = build_maintenance_notes(
            profile["hottest_month_avg_high_f"],
            profile["coldest_month_avg_low_f"],
            profile["annual_precip_in"],
        )
        add_trace("Climate profile received")
    except Exception as exc:
        profile["warning"] = f"Climate lookup failed: {exc}"
        add_trace(profile["warning"])

    LAST_CLIMATE_PROFILE = profile
    return profile


baseline_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are an HVAC product expert helping with a demo. Answer directly, but do not pretend to have access to internal documents or unseen product manuals.",
    ),
    ("human", "Question: {query}"),
])

rag_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are an HVAC product expert. Answer only from the retrieved evidence. If the evidence is incomplete, say so clearly. Cite filenames in square brackets such as [manual.pdf].",
    ),
    (
        "human",
        "Question:\n{query}\n\nRetrieved evidence:\n{evidence_text}\n\nReturn a grounded answer with a short evidence sufficiency note.",
    ),
])

AGENT_SYSTEM_PROMPT = """
You are an HVAC documentation analyst.

You have two tools:
- `get_climate_profile`: use this when the user query includes an explicit location or clearly climate-sensitive requirements tied to a named place
- `search_hvac_docs_tool`: use this for product facts, model comparisons, options, controls, monitoring, and recommendations

Decision rules:
- Inspect the user query and decide whether location or climate materially affects the answer.
- If the query includes an explicit location, first call `get_climate_profile` with the location text you infer from the query.
- After climate lookup, call `search_hvac_docs_tool` to ground any product recommendation in HVAC documents.
- If the query does not include a meaningful location, skip climate lookup and use document retrieval directly.

Answering rules:
- Climate is advisory only. Do not use climate data as a substitute for product evidence.
- Ground product claims in retrieved HVAC documentation and cite filenames in square brackets.
- If the retrieved docs do not support a specific model recommendation, say so plainly.
- If climate lookup fails, continue with retrieval-backed reasoning and mention that climate context was unavailable.
- When climate context is available and relevant, include a brief climate-fit rationale and maintenance considerations.
- Do not reveal chain-of-thought. Provide the final answer only.
""".strip()


def run_baseline(query: str):
    add_trace("Baseline LLM mode started")
    response = baseline_llm.invoke(baseline_prompt.format_messages(query=query))
    add_trace("Baseline LLM mode finished")
    return {"answer": message_to_text(response), "evidence": [], "trace": TRACE_LOG.copy(), "warning": "", "climate_profile": None}


def run_rag(query: str, top_k: int):
    add_trace("RAG mode started")
    retrieval = search_hvac_docs(query, top_k=top_k)
    if retrieval["results"]:
        response = rag_llm.invoke(rag_prompt.format_messages(query=query, evidence_text=retrieval["evidence_text"]))
        answer = message_to_text(response)
    else:
        answer = "I could not ground an answer because no supporting HVAC snippets were retrieved from the configured vector store."
    add_trace("RAG mode finished")
    return {
        "answer": answer,
        "evidence": retrieval["results"],
        "trace": TRACE_LOG.copy(),
        "warning": retrieval["warning"],
        "climate_profile": None,
    }


def run_deep_agent(query: str, top_k: int):
    global LAST_CLIMATE_PROFILE
    LAST_CLIMATE_PROFILE = None
    tool_evidence = []

    def search_hvac_docs_tool(tool_query: str) -> str:
        """Search the OpenAI-hosted HVAC vector store and return the most relevant document chunks as JSON."""
        retrieval = search_hvac_docs(tool_query, top_k=top_k)
        tool_evidence.extend(retrieval["results"])
        return json.dumps({
            "query": tool_query,
            "warning": retrieval["warning"],
            "results": retrieval["results"],
        }, indent=2)

    add_trace("Deep Agent mode started")
    agent = create_deep_agent(
        model=AGENT_MODEL,
        tools=[get_climate_profile, search_hvac_docs_tool],
        system_prompt=AGENT_SYSTEM_PROMPT,
    )
    result = agent.invoke({"messages": [{"role": "user", "content": query}]})
    answer = message_to_text(result["messages"][-1])
    add_trace("Deep Agent mode finished")

    unique_evidence = []
    seen = set()
    for item in tool_evidence:
        key = (item.get("file_id"), item.get("snippet"))
        if key in seen:
            continue
        seen.add(key)
        unique_evidence.append(item)

    warnings = []
    if not unique_evidence and OPENAI_VECTOR_STORE_ID == "vs_your_vector_store_id":
        warnings.append("OPENAI_VECTOR_STORE_ID is still a placeholder, so the agent could not retrieve evidence.")
    if LAST_CLIMATE_PROFILE and LAST_CLIMATE_PROFILE.get("warning"):
        warnings.append(LAST_CLIMATE_PROFILE["warning"])

    return {
        "answer": answer,
        "evidence": unique_evidence,
        "trace": TRACE_LOG.copy(),
        "warning": "\n".join(warnings),
        "climate_profile": LAST_CLIMATE_PROFILE,
    }


def render_evidence(evidence):
    if not evidence:
        display(Markdown("### Retrieved Evidence\n_No retrieved evidence for this mode._"))
        return

    display(Markdown("### Retrieved Evidence"))
    evidence_df = pd.DataFrame([
        {
            "filename": item["filename"],
            "score": item["score"],
            "file_id": item["file_id"],
            "snippet": textwrap.shorten(item["snippet"].replace("\n", " "), width=180, placeholder=" ..."),
        }
        for item in evidence
    ])
    display(evidence_df)

    details = []
    for idx, item in enumerate(evidence, start=1):
        snippet = item["snippet"] or "[No snippet returned]"
        details.append(f"**{idx}. {item['filename']}**  \nscore={item['score']} | file_id={item['file_id']}\n\n> {snippet.replace(chr(10), chr(10) + '> ')}")
    display(Markdown("\n\n".join(details)))


def render_climate_profile(climate_profile):
    if not climate_profile:
        return
    display(Markdown("### Climate Profile"))
    if climate_profile.get("warning") and not climate_profile.get("resolved_location"):
        display(Markdown(f"> {climate_profile['warning']}"))
        return

    summary_lines = [
        f"- Resolved location: {climate_profile.get('resolved_location', 'N/A')}",
        f"- Coordinates: {climate_profile.get('lat')} , {climate_profile.get('lon')}",
        f"- Typical annual high / low: {climate_profile.get('annual_typical_high_f')} F / {climate_profile.get('annual_typical_low_f')} F",
        f"- Hottest-month average high: {climate_profile.get('hottest_month_avg_high_f')} F",
        f"- Coldest-month average low: {climate_profile.get('coldest_month_avg_low_f')} F",
    ]
    if climate_profile.get("annual_precip_in") is not None:
        summary_lines.append(f"- Average annual precipitation: {climate_profile['annual_precip_in']} in")
    display(Markdown("\n".join(summary_lines)))

    if climate_profile.get("climate_notes"):
        display(Markdown("**Climate Notes**\n" + "\n".join(f"- {note}" for note in climate_profile["climate_notes"])))
    if climate_profile.get("maintenance_notes"):
        display(Markdown("**Maintenance Notes**\n" + "\n".join(f"- {note}" for note in climate_profile["maintenance_notes"])))
    if climate_profile.get("warning"):
        display(Markdown(f"> {climate_profile['warning']}"))


def render_trace(trace):
    if not trace:
        display(Markdown("### Agent Steps\n_No trace captured._"))
        return
    bullet_lines = "\n".join(f"- {line}" for line in trace)
    display(Markdown(f"### Agent Steps\n{bullet_lines}"))


def render_result(mode_label: str, query: str, result: dict):
    display(Markdown(f"## {mode_label}"))
    display(Markdown(f"**User Query**\n\n{query}"))
    if result.get("warning"):
        display(Markdown(f"> {result['warning']}"))
    render_evidence(result.get("evidence", []))
    render_climate_profile(result.get("climate_profile"))
    render_trace(result.get("trace", []))
    display(Markdown(f"### Final Answer\n{result.get('answer', '').strip()}"))


mode_widget = widgets.Dropdown(
    options=[("All modes", "all"), ("Baseline only", "baseline"), ("RAG only", "rag"), ("Deep Agent only", "deep_agent")],
    value="all",
    description="Mode:",
    layout=widgets.Layout(width="320px"),
)

query_widget = widgets.Textarea(
    value="Which rooftop unit supports BACnet/IP?",
    description="Query:",
    layout=widgets.Layout(width="100%", height="120px"),
)

top_k_widget = widgets.IntSlider(value=DEFAULT_TOP_K, min=1, max=8, step=1, description="Top K:", continuous_update=False)
run_button = widgets.Button(description="Run Demo", button_style="primary")
output = widgets.Output()


def on_run_clicked(_):
    with output:
        clear_output(wait=True)
        TRACE_LOG.clear()
        try:
            query = query_widget.value.strip()
            if not query:
                raise ValueError("Enter a query before running the demo.")
            top_k = int(top_k_widget.value)
            display(Markdown("# HVAC Demo Output"))
            display(Markdown(f"**Selected Mode**: `{mode_widget.value}`"))

            if mode_widget.value in ("all", "baseline"):
                TRACE_LOG.clear()
                render_result("Baseline LLM", query, run_baseline(query))

            if mode_widget.value in ("all", "rag"):
                TRACE_LOG.clear()
                render_result("RAG", query, run_rag(query, top_k=top_k))

            if mode_widget.value in ("all", "deep_agent"):
                TRACE_LOG.clear()
                render_result("Deep Agent + RAG", query, run_deep_agent(query, top_k=top_k))
        except Exception as exc:
            display(Markdown(f"## Error\n`{exc}`"))


run_button.on_click(on_run_clicked)

display(Markdown("# HVAC LangChain Deep Agent Demo"))
display(Markdown("Enter a single free-form query. The Deep Agent will infer whether the query includes a location, call the climate tool first when appropriate, and then retrieve HVAC documentation."))
display(widgets.VBox([
    widgets.HBox([mode_widget, top_k_widget]),
    query_widget,
    run_button,
    output,
]))

In [ ]:
SAMPLE_QUERIES = [
    "Which rooftop unit supports BACnet/IP?",
    "Compare Product X and Product Y for economizer support and remote monitoring in Phoenix, Arizona.",
    "Which chiller model do you recommend for a residence in Phoenix Arizona?",
]

for idx, query in enumerate(SAMPLE_QUERIES, start=1):
    print(f"{idx}. {query}")

## Notes For Extension

- Add richer metadata filters later if the HVAC corpus includes stable product-family metadata.
- Port the retrieval helpers and rendering logic into Streamlit later with minimal changes.
- If you want a stricter grounded mode, tighten the RAG prompt so it refuses to answer unless retrieval returns evidence.
- If you want richer Deep Agent traces, add LangSmith or LangGraph tracing later without exposing chain-of-thought.
- If you want broader climate decision support later, add humidity or degree-day metrics while keeping the query-driven climate step.